# 🎬 LTX-Video API — Colab Edition
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/Saqo12/ltx-video-api/blob/main/ltx_video_api.ipynb)

**What this does:**
1. Install LTX-Video & dependencies
2. Download the model (~3GB)
3. Start a local API server on port 7860
4. Expose it via ngrok tunnel
5. Print your API URL — paste it into your TikTok pipeline config

⚠️ **Keep this tab open!** Colab sessions last ~12hrs. Re-run to regenerate.

---


## 1. Install dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/Lightricks/LTX-Video.git
!pip install -q fastapi uvicorn pyngrok python-multipart pillow
!pip install -q transformers accelerate huggingface_hub

print('✅ Dependencies installed')

## 2. Download model

In [ ]:
import os
os.makedirs('/root/.cache/huggingface/hub', exist_ok=True)

from huggingface_hub import snapshot_download
print('Downloading LTX-Video model (may take 5-10 mins)...')
snapshot_download(
    repo_id='Lightricks/LTX-Video',
    cache_dir='/root/.cache/huggingface/hub',
    local_dir='/root/.cache/huggingface/hub/LTX-Video',
)
print('✅ Model ready')

## 3. API Server

In [ ]:
%%writefile /content/ltx_api.py
import os, io, base64, uuid
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn
import torch
from PIL import Image

# Suppress warnings
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

app = FastAPI(title='LTX-Video API')

@app.on_event('startup')
async def startup():
    global pipe
    from ltх_video.pipeline import LTXVideoPipeline
    print('Loading LTX-Video model...')
    pipe = LTXVideoPipeline.from_pretrained(
        '/root/.cache/huggingface/hub/LTX-Video',
        torch_dtype=torch.float16,
        device='cuda',
    )
    print('✅ Model loaded — GPU ready!')

@app.get('/')
async def root():
    return {'status': 'ok', 'model': 'LTX-Video', 'gpu': torch.cuda.is_available()}

@app.get('/health')
async def health():
    return {'ok': True}

@app.post('/generate')
async def generate(
    prompt: str = Form(...),
    image: UploadFile = File(None),
    duration_seconds: int = Form(4),
    width: int = Form(512),
    height: int = Form(512),
):
    global pipe
    job_id = str(uuid.uuid4())[:8]
    print(f'[{job_id}] Generating: {prompt[:60]}...')
    
    kwargs = {
        'prompt': prompt,
        'num_frames': duration_seconds * 8,
        'width': width,
        'height': height,
        'num_inference_steps': 30,
        'guidance_scale': 3.5,
    }
    
    if image:
        img = Image.open(io.BytesIO(await image.read())).convert('RGB')
        kwargs['image'] = img
    
    output = pipe(**kwargs)
    frames = output.frames[0]
    
    # Save as MP4
    import tempfile, subprocess
    with tempfile.TemporaryDirectory() as tmpdir:
        video_path = f'{tmpdir}/output.mp4'
        import numpy as np
        frames_np = (np.array(f) for f in frames)
        
        from PIL import Image
        imgs = [Image.fromarray(f) for f in frames]
        imgs[0].save(
            video_path,
            save_all=True,
            append_images=imgs[1:],
            duration=int(1000/24),
            loop=0,
            fps=24,
            plugin=['pyimagedice']
        )
        with open(video_path, 'rb') as f:
            video_data = f.read()
    
    b64 = base64.b64encode(video_data).decode()
    print(f'[{job_id}] Done! {len(video_data)//1024}KB')
    return JSONResponse({'job_id': job_id, 'video': b64, 'size_kb': len(video_data)//1024})

# Start tunnel + server
public_url = ngrok.connect(7860, bind_tls=True).url
print(f'\n🎬 LTX-Video API: {public_url}')
print(f'📌 Use this URL in your TikTok pipeline config\n')

uvicorn.run(app, host='0.0.0.0', port=7860, log_level='warning')

## 4. 🚀 Run this cell to start!

After running, copy the ngrok URL above and paste it into your config.

In [ ]:
# !Authenticate ngrok (get your token from https://dashboard.ngrok.com/auth)
# !ngrok config add-authtoken YOUR_NGROK_TOKEN
# Run the API server (keep cell running)
# %tb to show traceback
!python /content/ltx_api.py